# Addestramento e valutazione del modello

Classifica la **perdita interna della pompa** (0 nessuna, 1 debole, 2 grave) per ogni ciclo del dataset UCI
*Condition Monitoring of Hydraulic Systems*.

Produce `model.joblib` (usato da `worker.py`), `cicli_test.txt` (workload fisso per gli esperimenti)
e `risultati/valutazione_modello.txt`.

In [ ]:
import joblib
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import comune

## 1. Dati

Scarico lo zip una volta sola (~70 MB). Ogni file di sensore ha **una riga = un ciclo di 60 s**.
L'etichetta e' la colonna 3 di `profile.txt`.

In [ ]:
comune.scarica_dataset()
misure, y = comune.carica_misure(), comune.etichette_vere()
print("cicli:", len(y))
print("distribuzione classi:", {comune.ETICHETTE[k]: int(v) for k, v in zip(*np.unique(y, return_counts=True))})

## 2. Feature

Ogni sensore viene riassunto con media, deviazione standard, minimo e massimo:
17 sensori x 4 = **68 numeri per ciclo**. L'etichetta vera **non** entra mai nelle feature.

In [ ]:
X = np.array([comune.feature(misure, i) for i in range(len(y))])
X.shape

## 3. Divisione train / validation / test

60 / 20 / 20, stratificata per classe. Una riga e' un ciclo intero, quindi tutti i campioni
di un ciclo finiscono nella stessa partizione.

In [ ]:
indici = np.arange(len(y))
tr, resto = train_test_split(indici, test_size=0.4, stratify=y, random_state=0)
va, te = train_test_split(resto, test_size=0.5, stratify=y[resto], random_state=0)
print(f"train {len(tr)}  validation {len(va)}  test {len(te)}")

## 4. Modelli a confronto

- **riferimento**: predice sempre la classe piu' frequente (il minimo da battere)
- **regressione logistica** e **random forest**

Lo scaler sta dentro la pipeline, quindi viene fittato **solo sui dati di training**.
Il modello si sceglie guardando solo la validation.

In [ ]:
modelli = {
    "riferimento (classe piu' frequente)": DummyClassifier(strategy="most_frequent"),
    "regressione logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "random forest": make_pipeline(StandardScaler(), RandomForestClassifier(200, random_state=0, n_jobs=-1)),
}
righe = [f"cicli: train {len(tr)}  validation {len(va)}  test {len(te)}", "", "F1 macro su validation:"]
f1_val = {}
for nome, m in modelli.items():
    m.fit(X[tr], y[tr])
    f1_val[nome] = f1_score(y[va], m.predict(X[va]), average="macro")
    righe.append(f"  {nome:38s} {f1_val[nome]:.3f}")

migliore = max((n for n in modelli if not n.startswith("riferimento")), key=f1_val.get)
righe += ["", f"scelto su validation: {migliore}", ""]
print("\n".join(righe))

## 5. Valutazione sul test

Il test set si usa **una volta sola**, per il modello scelto e per il riferimento:
matrice di confusione e precision / recall / F1 per classe.

In [ ]:
for nome in ("riferimento (classe piu' frequente)", migliore):
    pred = modelli[nome].predict(X[te])
    blocco = [f"=== TEST: {nome}", "matrice di confusione (righe = vero 0/1/2, colonne = previsto):",
              str(confusion_matrix(y[te], pred, labels=[0, 1, 2])),
              classification_report(y[te], pred, labels=[0, 1, 2],
                                    target_names=list(comune.ETICHETTE.values()), zero_division=0)]
    righe += blocco
    print("\n".join(blocco))

### Limiti della valutazione

- I cicli vicini nel tempo si somigliano: la divisione casuale puo' rendere il test "facile"
  (un punteggio perfetto va preso con cautela; una divisione per blocchi temporali sarebbe piu' severa).
- Un solo banco prova in laboratorio: i risultati non si trasferiscono a pompe reali senza nuovi dati.
- Il test ha 441 cicli: anche un piccolo numero di errori cambierebbe molto le metriche delle classi meno numerose.

## 6. Salvataggio

Un core per worker (`n_jobs=1`): cosi' l'esperimento di scalabilita' misura i worker e non i thread di sklearn.

In [ ]:
m = modelli[migliore]
m.set_params(**{k: 1 for k in m.get_params() if k.endswith("n_jobs")})
joblib.dump(m, comune.CARTELLA / "model.joblib")
np.savetxt(comune.CARTELLA / "cicli_test.txt", te, fmt="%d")
(comune.CARTELLA / "risultati").mkdir(exist_ok=True)
(comune.CARTELLA / "risultati" / "valutazione_modello.txt").write_text("\n".join(righe), encoding="utf-8")
print("salvati model.joblib, cicli_test.txt, risultati/valutazione_modello.txt")